## <a name="roi_align">5. RoI Align</a>

### 为什么要引入RoI Align

RoI Align的提出，主要是为了弥补Faster R-CNN中，RoI Pooling精度丢失的不足。在Faster R-CNN中，backbone网络使用的是VGG16，生成的特征图stride为32，即特征图的宽高为原图1/32。因此，原图的包围框在向特征图映射时，宽高除以32，会出现浮点数，Faster R-CNN中的处理方式是直接将浮点数进行取整操作，这一操作损失了精度。为了后续的分类网络能够对包围框进行回归，特征图上的包围框要输入到RoI Pooling层，进行池化操作，将宽度和高度固定为7x7，缩放后还会出现浮点数，RoI Pooling层再进行一次取整操作，损失一部分精度。经过两次取整，对于较小的目标，会出现比较大的精度损失。整个RoI Pooling的操作过程如下图所示：

![roi_pooling](img/roi_pooling.jpg)

### RoI Align的改进

在Mask R-CNN中，RPN中生成的Proposal同样需要进行池化操作，将Proposal的包围框大小固定，用于后续head网络的训练。与Faster R-CNN不同，我们使用ResNet+FPN作为backbone网络，输出的特征图共有5个层次，即[P2, P3, P4, P5, P6]，在RPN中我们针对所有的Level都生成了Anchor，因此，首先我们需要确定Proposal的包围框，在特征图上对应哪一个层次。在FPN论文中，作者提出下面的层级计算公式：

$$k=[k_0 + log_2(\frac {\sqrt{wh}} {224})]$$

其中，$w$和$h$分别为Anchor的宽度和高度，224是ImageNet预训练的尺寸，$k_0$是$wh=224^2$$时所在的层级，一般取为4。得到Anchor所在的层级，我们就得到该层级的stride。之后我们再进行$$原图 \to 特征图 \to RoI$池化操作的过程。在两次缩放的过程中，我们不进行取整操作，仍然保留浮点数。以上图中的包围框为例，假设FPN所在层的stride仍然为32，则：

$$ \frac{(665x665)} {32} = (20.78x20.78), 20.78 \div 7 = 2.97 $$

因此，特征图映射到Pooling时,(2.97x2.97)成为输出的像素单位，这个单位的值如何计算呢？我们把(2.97x2.97)的区域平均分为4份，然后计算4个区域中心点的像素值，这四个中心点的像素值是通过[双线性内插值算法]()计算出来的，得到这4个值之后，再进行一次池化操作，得到池化层输出结果的一个单位的像素值。这个计算过程如下图所示：

![roi_align_process](img/roi_align_process.jpg)

这样RoI Align在得到固定尺寸的特征区域后，避免了精度损失带来的问题，更精确的保留了原图、特征图在池化层的映射。在代码实现中，RoI Align层将包围框的大小池化为7x7，Mask遮罩的大小池化为14x14。